In [ ]:
# ============================================
# Pixel-level feature extraction:
# Color (LAB, XYZ, RGB, CMYK)
# Sieve-like particle size features
# Texture (GLCM) + Morphological surrogates
# ============================================

!pip install -q scikit-image pandas

import os
from glob import glob
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

from skimage.feature import graycomatrix, graycoprops
from skimage.measure import label, regionprops

# ------------------------------------------------
# 1. CONFIG
# ------------------------------------------------
# Folder with original RGB images
IMAGE_DIR = "/content/drive/MyDrive/CompostDataset/z_all"  

# Folder with corresponding binary masks (GT or predicted)
MASK_DIR  = "/content/drive/MyDrive/CompostDataset/z_all_masks"   

# Output CSV
OUT_CSV   = "/content/drive/MyDrive/Compost_features/compost_features_full.csv"
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)

# Pixel size calibration (mm per pixel) for sieve-like features.
PIXEL_SIZE_MM = 0.1

# Sieve bin thresholds in mm
SIEVE_THRESHOLDS_MM = {
    "Sieve_19mm_min": 19.0,
    "Sieve_9_5mm_min": 9.5,
    "Sieve_4_75mm_min": 4.75,
    "Sieve_2_36mm_min": 2.36,
    # Pan will be < 2.36mm
}

# ------------------------------------------------
# 2. Pair images and masks by filename stem
# ------------------------------------------------
def make_pairs(image_dir, mask_dir):
    img_paths = sorted(glob(os.path.join(image_dir, "*")))
    msk_paths = sorted(glob(os.path.join(mask_dir, "*")))

    img_map = {Path(p).stem: p for p in img_paths}

    mask_map = {}
    for p in msk_paths:
        stem = Path(p).stem
        # allow names like "XYZ_mask.png"
        if stem.endswith("_mask"):
            stem = stem[:-5]
        mask_map[stem] = p

    common = sorted(set(img_map.keys()).intersection(mask_map.keys()))
    pairs = [(img_map[k], mask_map[k]) for k in common]
    if not pairs:
        raise RuntimeError("No matching image–mask pairs found. Check IMAGE_DIR and MASK_DIR.")
    print(f"Found {len(pairs)} image–mask pairs.")
    return pairs

pairs = make_pairs(IMAGE_DIR, MASK_DIR)

# ------------------------------------------------
# 3. COLOR FEATURE HELPERS (LAB, XYZ, RGB, CMYK)
# ------------------------------------------------
def rgb_to_cmyk(r, g, b):
    """
    r, g, b in [0,255] arrays over masked pixels.
    Returns C,M,Y,K in [0,1].
    """
    r = r.astype(np.float32) / 255.0
    g = g.astype(np.float32) / 255.0
    b = b.astype(np.float32) / 255.0

    K = 1.0 - np.maximum.reduce([r, g, b])
    # Avoid division by zero
    denom = (1.0 - K)
    denom[denom == 0] = 1.0

    C = (1.0 - r - K) / denom
    M = (1.0 - g - K) / denom
    Y = (1.0 - b - K) / denom

    # If pixel is pure black (K=1), set C=M=Y=0
    C[K >= 0.999] = 0.0
    M[K >= 0.999] = 0.0
    Y[K >= 0.999] = 0.0

    return C, M, Y, K

def extract_color_features_all(bgr, mask_bin):
    """
    bgr: HxWx3, uint8 (OpenCV format)
    mask_bin: HxW, 0/1 (compost region)
    Returns dict of mean/std for LAB, XYZ, RGB, CMYK over masked pixels.
    """
    mask_bool = mask_bin.astype(bool)
    if mask_bool.sum() == 0:
        # no compost region -> NaNs
        return {k: np.nan for k in [
            # LAB
            "L_mean","a_mean","b_mean","L_std","a_std","b_std",
            # XYZ
            "X_mean","Y_mean","Z_mean","X_std","Y_std","Z_std",
            # RGB
            "R_mean","G_mean","B_mean","R_std","G_std","B_std",
            # CMYK
            "C_mean","M_mean","Yc_mean","K_mean",
            "C_std","M_std","Yc_std","K_std"
        ]}

    # BGR -> RGB
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    R = rgb[:,:,0][mask_bool].astype(np.float32)
    G = rgb[:,:,1][mask_bool].astype(np.float32)
    B = rgb[:,:,2][mask_bool].astype(np.float32)

    # LAB (OpenCV expects BGR)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    L = lab[:,:,0][mask_bool].astype(np.float32)
    a = lab[:,:,1][mask_bool].astype(np.float32)
    b = lab[:,:,2][mask_bool].astype(np.float32)

    # XYZ (OpenCV conversion)
    xyz = cv2.cvtColor(bgr, cv2.COLOR_BGR2XYZ)
    X = xyz[:,:,0][mask_bool].astype(np.float32)
    Y = xyz[:,:,1][mask_bool].astype(np.float32)
    Z = xyz[:,:,2][mask_bool].astype(np.float32)

    # CMYK from RGB
    C, M, Yc, K = rgb_to_cmyk(R, G, B)

    feats = {
        # LAB
        "L_mean": float(L.mean()), "a_mean": float(a.mean()), "b_mean": float(b.mean()),
        "L_std":  float(L.std()),  "a_std":  float(a.std()),  "b_std":  float(b.std()),
        # XYZ
        "X_mean": float(X.mean()), "Y_mean": float(Y.mean()), "Z_mean": float(Z.mean()),
        "X_std":  float(X.std()),  "Y_std":  float(Y.std()),  "Z_std":  float(Z.std()),
        # RGB
        "R_mean": float(R.mean()), "G_mean": float(G.mean()), "B_mean": float(B.mean()),
        "R_std":  float(R.std()),  "G_std":  float(G.std()),  "B_std":  float(B.std()),
        # CMYK
        "C_mean":  float(C.mean()),  "M_mean":  float(M.mean()),
        "Yc_mean": float(Yc.mean()), "K_mean":  float(K.mean()),
        "C_std":   float(C.std()),   "M_std":   float(M.std()),
        "Yc_std":  float(Yc.std()),  "K_std":   float(K.std()),
    }
    return feats

# ------------------------------------------------
# 4. TEXTURE (GLCM) FEATURES
# ------------------------------------------------
def extract_texture_features_glcm(bgr, mask_bin, levels=32):
    """
    Compute GLCM-based texture over compost region.
    We zero-out background before GLCM.
    """
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

    # zero-out background
    gray_masked = gray.copy()
    gray_masked[mask_bin == 0] = 0

    # quantize grayscale to `levels`
    gray_q = (gray_masked.astype(np.float32) / 255.0 * (levels-1)).astype(np.uint8)

    distances = [1, 2, 4]
    angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]

    glcm = graycomatrix(
        gray_q,
        distances=distances,
        angles=angles,
        levels=levels,
        symmetric=True,
        normed=True
    )

    feats = {}
    for prop in ["contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"]:
        vals = graycoprops(glcm, prop)  # shape (len(distances), len(angles))
        feats[f"glcm_{prop}"] = float(vals.mean())

    return feats

# ------------------------------------------------
# 5. SIEVE-LIKE PARTICLE SIZE + MORPHOLOGICAL FEATURES
# ------------------------------------------------
def extract_particle_features(mask_bin, pixel_size_mm=PIXEL_SIZE_MM):
    """
    mask_bin: HxW 0/1 compost mask.
    1. Finds connected components = particles.
    2. For each particle: area (pixels) and equivalent diameter (in mm).
    3. Aggregates:
       - #particles
       - mean/std area (pixels)
       - mean/std equivalent diameter (mm)
       - mean solidity, extent, eccentricity (shape surrogates)
       - sieve-like distribution: % compost area in each size bin.
    """
    bin_uint8 = (mask_bin > 0).astype(np.uint8)
    total_area_pixels = bin_uint8.sum()

    if total_area_pixels == 0:
        # no compost
        feats = {
            "total_area_pixels": 0,
            "n_particles": 0,
            "particle_area_mean": np.nan,
            "particle_area_std": np.nan,
            "equiv_diam_mm_mean": np.nan,
            "equiv_diam_mm_std": np.nan,
            "solidity_mean": np.nan,
            "extent_mean": np.nan,
            "eccentricity_mean": np.nan,
        }
        # sieve %
        for name in ["Sieve_19mm", "Sieve_9_5mm", "Sieve_4_75mm", "Sieve_2_36mm", "Sieve_Pan"]:
            feats[f"{name}_area_frac"] = np.nan
        return feats

    # label connected components
    labeled = label(bin_uint8, connectivity=2)
    regions = regionprops(labeled)

    areas = []
    equiv_diams_mm = []
    solidities = []
    extents = []
    eccentricities = []

    # accumulate area per sieve bin (pixel area, converted to fraction later)
    sieve_area_bins = {
        "Sieve_19mm": 0.0,
        "Sieve_9_5mm": 0.0,
        "Sieve_4_75mm": 0.0,
        "Sieve_2_36mm": 0.0,
        "Sieve_Pan": 0.0
    }

    for r in regions:
        A = float(r.area)   # pixels
        if A <= 0:
            continue
        areas.append(A)
        eq_diam_pixels = 2.0 * np.sqrt(A / np.pi)
        # convert to mm if pixel_size_mm is set
        eq_diam_mm = eq_diam_pixels * pixel_size_mm
        equiv_diams_mm.append(eq_diam_mm)

        solidities.append(float(r.solidity))
        extents.append(float(r.extent))
        eccentricities.append(float(r.eccentricity))

        # Assign to sieve bin based on equivalent diameter in mm
        d = eq_diam_mm
        if d >= SIEVE_THRESHOLDS_MM["Sieve_19mm_min"]:
            sieve_area_bins["Sieve_19mm"] += A
        elif d >= SIEVE_THRESHOLDS_MM["Sieve_9_5mm_min"]:
            sieve_area_bins["Sieve_9_5mm"] += A
        elif d >= SIEVE_THRESHOLDS_MM["Sieve_4_75mm_min"]:
            sieve_area_bins["Sieve_4_75mm"] += A
        elif d >= SIEVE_THRESHOLDS_MM["Sieve_2_36mm_min"]:
            sieve_area_bins["Sieve_2_36mm"] += A
        else:
            sieve_area_bins["Sieve_Pan"] += A

    areas = np.array(areas, dtype=np.float32)
    equiv_diams_mm = np.array(equiv_diams_mm, dtype=np.float32)
    solidities = np.array(solidities, dtype=np.float32)
    extents = np.array(extents, dtype=np.float32)
    eccentricities = np.array(eccentricities, dtype=np.float32)

    feats = {
        "total_area_pixels": float(total_area_pixels),
        "n_particles": int(len(areas)),
        "particle_area_mean": float(areas.mean()) if len(areas) > 0 else np.nan,
        "particle_area_std": float(areas.std()) if len(areas) > 0 else np.nan,
        "equiv_diam_mm_mean": float(equiv_diams_mm.mean()) if len(equiv_diams_mm) > 0 else np.nan,
        "equiv_diam_mm_std": float(equiv_diams_mm.std()) if len(equiv_diams_mm) > 0 else np.nan,
        "solidity_mean": float(solidities.mean()) if len(solidities) > 0 else np.nan,
        "extent_mean": float(extents.mean()) if len(extents) > 0 else np.nan,
        "eccentricity_mean": float(eccentricities.mean()) if len(eccentricities) > 0 else np.nan,
    }

    # convert sieve bin areas to fraction of total compost area
    for name, area_bin in sieve_area_bins.items():
        feats[f"{name}_area_frac"] = float(area_bin / total_area_pixels)

    return feats

# ------------------------------------------------
# 6. MAIN LOOP: EXTRACT FEATURES PER IMAGE
# ------------------------------------------------
rows = []

for img_path, mask_path in tqdm(pairs, desc="Extracting features"):
    img_name = Path(img_path).name

    # read image (BGR)
    bgr = cv2.imread(img_path)
    if bgr is None:
        print(f"WARNING: Could not read image {img_path}, skipping.")
        continue

    # read mask
    m = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
    if m is None:
        print(f"WARNING: Could not read mask {mask_path}, skipping.")
        continue

    if m.ndim == 3:
        mask_bin = (m.sum(axis=2) > 0).astype(np.uint8)
    else:
        mask_bin = (m > 0).astype(np.uint8)

    # Ensure mask and image have same size
    if mask_bin.shape[:2] != bgr.shape[:2]:
        mask_bin = cv2.resize(mask_bin.astype(np.uint8),
                              (bgr.shape[1], bgr.shape[0]),
                              interpolation=cv2.INTER_NEAREST)
        mask_bin = (mask_bin > 0).astype(np.uint8)

    # ---- Extract features ----
    color_feats = extract_color_features_all(bgr, mask_bin)
    tex_feats = extract_texture_features_glcm(bgr, mask_bin, levels=32)
    particle_feats = extract_particle_features(mask_bin, pixel_size_mm=PIXEL_SIZE_MM)

    row = {
        "image": img_name,
        "image_path": img_path,
        "mask_path": mask_path,
    }
    row.update(color_feats)
    row.update(tex_feats)
    row.update(particle_feats)
    rows.append(row)

# ------------------------------------------------
# 7. Save to CSV
# ------------------------------------------------
df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)
print("Saved features to:", OUT_CSV)
print("Feature matrix shape:", df.shape)
df.head()


Found 229 image–mask pairs.


Extracting features: 100%|██████████| 229/229 [20:01<00:00,  5.25s/it]

Saved features to: /content/drive/MyDrive/Compost_features/compost_features_full.csv
Feature matrix shape: (229, 49)


,image,image_path,mask_path,L_mean,a_mean,b_mean,L_std,a_std,b_std,X_mean,...,equiv_diam_mm_mean,equiv_diam_mm_std,solidity_mean,extent_mean,eccentricity_mean,Sieve_19mm_area_frac,Sieve_9_5mm_area_frac,Sieve_4_75mm_area_frac,Sieve_2_36mm_area_frac,Sieve_Pan_area_frac
0,HA-1.jpeg,/content/drive/MyDrive/CompostDataset/z_all/HA...,/content/drive/MyDrive/CompostDataset/z_all_ma...,81.689293,129.490448,140.434601,41.981674,1.338305,5.202976,72.647575,...,241.423630,0.0,0.933096,0.816666,0.838516,1.0,0.0,0.0,0.0,0.0
1,HA-10.jpeg,/content/drive/MyDrive/CompostDataset/z_all/HA...,/content/drive/MyDrive/CompostDataset/z_all_ma...,75.293541,129.210022,138.728241,41.868099,1.271718,4.819153,67.131859,...,248.253693,0.0,0.933008,0.819510,0.827540,1.0,0.0,0.0,0.0,0.0
2,HA-2.jpeg,/content/drive/MyDrive/CompostDataset/z_all/HA...,/content/drive/MyDrive/CompostDataset/z_all_ma...,80.089600,129.459976,139.692749,41.771935,1.330944,5.098574,71.303650,...,239.584335,0.0,0.933842,0.785836,0.838387,1.0,0.0,0.0,0.0,0.0
3,HA-3.jpeg,/content/drive/MyDrive/CompostDataset/z_all/HA...,/content/drive/MyDrive/CompostDataset/z_all_ma...,79.596466,129.685226,140.323807,41.762878,1.370379,5.231837,70.891685,...,250.738754,0.0,0.902665,0.781081,0.853999,1.0,0.0,0.0,0.0,0.0
4,HA-4.jpeg,/content/drive/MyDrive/CompostDataset/z_all/HA...,/content/drive/MyDrive/CompostDataset/z_all_ma...,80.216370,129.629578,139.703018,42.323284,1.352196,5.056995,71.508698,...,256.534729,0.0,0.939798,0.813048,0.836692,1.0,0.0,0.0,0.0,0.0


In [ ]:
print(df.columns.tolist())


['image', 'image_path', 'mask_path', 'L_mean', 'a_mean', 'b_mean', 'L_std', 'a_std', 'b_std', 'X_mean', 'Y_mean', 'Z_mean', 'X_std', 'Y_std', 'Z_std', 'R_mean', 'G_mean', 'B_mean', 'R_std', 'G_std', 'B_std', 'C_mean', 'M_mean', 'Yc_mean', 'K_mean', 'C_std', 'M_std', 'Yc_std', 'K_std', 'glcm_contrast', 'glcm_dissimilarity', 'glcm_homogeneity', 'glcm_energy', 'glcm_correlation', 'glcm_ASM', 'total_area_pixels', 'n_particles', 'particle_area_mean', 'particle_area_std', 'equiv_diam_mm_mean', 'equiv_diam_mm_std', 'solidity_mean', 'extent_mean', 'eccentricity_mean', 'Sieve_19mm_area_frac', 'Sieve_9_5mm_area_frac', 'Sieve_4_75mm_area_frac', 'Sieve_2_36mm_area_frac', 'Sieve_Pan_area_frac']


In [ ]:
import pandas as pd
hf=pd.read_csv('/compost_features_full_1.csv')

In [ ]:
import numpy as np
hf['c'] = np.sqrt(hf['a_mean']**2 + hf['b_mean']**2)
hf['h'] = np.degrees(np.arctan2(hf['b_mean'], hf['a_mean']))


In [ ]:
hf.to_csv('/compost_features_full_1.csv', index=False)

In [ ]:
test = pd.read_csv('/compost_features_full_1.csv')
print(test.columns)


Index(['Sample', 'image_path', 'mask_path', 'L_mean', 'a_mean', 'b_mean',
       'L_std', 'a_std', 'b_std', 'X_mean', 'Y_mean', 'Z_mean', 'X_std',
       'Y_std', 'Z_std', 'R_mean', 'G_mean', 'B_mean', 'R_std', 'G_std',
       'B_std', 'C_mean', 'M_mean', 'Yc_mean', 'K_mean', 'C_std', 'M_std',
       'Yc_std', 'K_std', 'glcm_contrast', 'glcm_dissimilarity',
       'glcm_homogeneity', 'glcm_energy', 'glcm_correlation', 'glcm_ASM',
       'total_area_pixels', 'n_particles', 'particle_area_mean',
       'particle_area_std', 'equiv_diam_mm_mean', 'equiv_diam_mm_std',
       'solidity_mean', 'extent_mean', 'eccentricity_mean',
       'Sieve_19mm_area_frac', 'Sieve_9_5mm_area_frac',
       'Sieve_4_75mm_area_frac', 'Sieve_2_36mm_area_frac',
       'Sieve_Pan_area_frac', 'c', 'h'],
      dtype='object')
